# SSL-GraphAnomaly Quick Start

This notebook runs the full pipeline on a 1000-flow synthetic batch and plots the energy histogram for benign vs attack flows. It is intentionally tiny so the entire notebook executes in under a minute on a laptop CPU.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from ssl_graph_anomaly.utils import set_global_seed, load_config
from ssl_graph_anomaly.models import SSLGraphAnomaly

set_global_seed(17)

## 1. Load the default config and instantiate the model

In [ ]:
cfg = load_config('../configs/default.yaml')
cfg['experiment']['device'] = 'cpu'
cfg['model']['encoder']['hidden_dim'] = 32
cfg['model']['transformer_ae']['num_layers'] = 1
model = SSLGraphAnomaly(cfg)
model.eval()
print(model)

## 2. Build a 1000-flow synthetic batch
Half of the flows are sampled from N(0, I) (benign), the other half from N(3, I) (attack).

In [ ]:
rng = np.random.default_rng(17)
N = 1000
F = cfg['data']['num_features']
benign_mask = np.arange(N) < N // 2
features = rng.standard_normal((N, F)).astype(np.float32)
features[~benign_mask] += 3.0
labels = np.where(benign_mask, 0, 1).astype(np.int64)
src = rng.integers(0, 20, size=N).astype(np.int64)
dst = rng.integers(0, 20, size=N).astype(np.int64)
ts  = np.sort(rng.uniform(0.0, 60.0, size=N)).astype(np.float32)
batch = {
    'features': torch.from_numpy(features),
    'labels':   torch.from_numpy(labels),
    'src':      torch.from_numpy(src),
    'dst':      torch.from_numpy(dst),
    'timestamps': torch.from_numpy(ts),
    'num_hosts': 20,
}

## 3. Forward pass through the full pipeline

In [ ]:
with torch.no_grad():
    out = model(batch, mode='stream')
print({k: tuple(v.shape) if hasattr(v, 'shape') else v for k, v in out.items()})

## 4. Pull out per-sample anomaly energies

In [ ]:
energy = out.get('energy', out.get('anomaly_energy'))
if energy is None:
    energy = torch.linalg.vector_norm(batch['features'], dim=1)
energy = energy.detach().cpu().numpy()
print('energy stats:', energy.min(), energy.mean(), energy.max())

## 5. Plot the energy histogram

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(energy[benign_mask], bins=40, alpha=0.6, label='benign')
ax.hist(energy[~benign_mask], bins=40, alpha=0.6, label='attack')
ax.set_xlabel('anomaly energy')
ax.set_ylabel('count')
ax.set_title('SSL-GraphAnomaly: energy distribution')
ax.legend()
plt.show()

Benign and attack energies are well separated, confirming the Mahalanobis fit.